### 일반 Attention vs Multi-Head Attention

(1) 같은 문장에서도 “관계”는 여러 종류라서
예: “나는 어제 은행에 갔다”
“은행”이 finance인지 river bank인지 문맥으로 판단해야 함
어떤 헤드는 “시간/장소 단서”에
다른 헤드는 “주변 단어 의미”에
또 다른 헤드는 “문장 전역 정보”에 집중하는 식으로 동시에 여러 관계를 잡아냄

(2) 긴 문장/복잡한 문맥에서 더 잘 버팀
싱글 attention은 전역을 다 보긴 하지만 “한 가지 정렬”로만 보니까,
복잡한 의존성이 많아질수록 한 번에 잡기 힘든데
MHA는 여러 헤드가 분산해서 잡아주니 안정적.

(3) 병렬 연산이 잘 맞아서(Transformer의 장점 극대화)
RNN처럼 순차가 아니라 행렬곱 중심이라 GPU에서 효율이 좋고,
MHA는 “여러 attention을 병렬로” 돌려도 구조적으로 잘 맞음.

In [1]:
# 일반 Attension 메커니즘
import torch
import torch.nn as nn
import torch.nn.functional as F

X = torch.tensor([[[1.0, 0.0, 1.0, 0.0],
                   [0.0, 2.0, 0.0, 2.0],
                   [1.0, 1.0, 1.0, 1.0]]])
print(f"X : {X.shape}")  # (B, T, F)

# Q, K, V를 생성하는 선형층
W_q = nn.Linear(4, 4, bias=False)  # Query 생성용 선형변환 (4 -> 4)
W_k = nn.Linear(4, 4, bias=False)  # Key 생성용 선형변환 (4 -> 4)
W_v = nn.Linear(4, 4, bias=False)  # Value 생성용 선형변환 (4 -> 4)

Q = W_q(X)  # X -> Q (배치, 길이, 차원)
K = W_k(X)
V = W_v(X)

print(f"Q : {Q.shape}")
print(f"K : {K.shape}")
print(f"V : {V.shape}")

# Q, K 유사도계산
attn_scores = torch.matmul(Q, K.transpose(-2, -1))  # Q•K^T 토큰간 유사도 계산
attn_scores /= Q.size(-1) ** 0.5  # 차원(d_k)로 나눈 후 score 스케일 조정(softmax값 안정화)
print(f"attn_scores : {attn_scores.shape}")

# Attention 분포 (확률)
attn_weights = F.softmax(attn_scores, dim=-1)  # 각 토큰이 바라볼 비율을 확률로 변환(행 단위 합)
print(f"attn_weights : {attn_weights.shape}")

# V-attention 분포의 가중합
output = torch.matmul(attn_weights, V)  # attention 가중치와 Value의 가중합 = 최종 출력
print(f"attn_value: {output.shape}")

X : torch.Size([1, 3, 4])
Q : torch.Size([1, 3, 4])
K : torch.Size([1, 3, 4])
V : torch.Size([1, 3, 4])
attn_scores : torch.Size([1, 3, 3])
attn_weights : torch.Size([1, 3, 3])
attn_value: torch.Size([1, 3, 4])


In [2]:
print(f"X : {X}")
print(f"Q : {Q}")
print(f"K : {K}")
print(f"V : {V}")

print(f'attention 분포 : {attn_weights}')
print(f'최종 출력 : {output}')

X : tensor([[[1., 0., 1., 0.],
         [0., 2., 0., 2.],
         [1., 1., 1., 1.]]])
Q : tensor([[[-0.0387,  0.2319,  0.4954,  0.6445],
         [-0.1507,  0.3329, -1.9820,  0.7180],
         [-0.1141,  0.3983, -0.4956,  1.0035]]], grad_fn=<UnsafeViewBackward0>)
K : tensor([[[-0.4048,  0.2637, -0.7199,  0.4053],
         [ 0.3715, -0.1744, -0.4475, -1.5805],
         [-0.2190,  0.1765, -0.9437, -0.3850]]], grad_fn=<UnsafeViewBackward0>)
V : tensor([[[ 0.1918, -0.1231, -0.1926, -0.2816],
         [ 0.6302, -0.7021,  0.4648, -0.2537],
         [ 0.5069, -0.4741,  0.0398, -0.4084]]], grad_fn=<UnsafeViewBackward0>)
attention 분포 : tensor([[[0.4441, 0.2346, 0.3213],
         [0.4461, 0.1464, 0.4075],
         [0.5015, 0.1518, 0.3467]]], grad_fn=<SoftmaxBackward0>)
최종 출력 : tensor([[[ 0.3959, -0.3717,  0.0363, -0.3158],
         [ 0.3844, -0.3509, -0.0017, -0.3292],
         [ 0.3676, -0.3326, -0.0123, -0.3213]]], grad_fn=<UnsafeViewBackward0>)


Attention 메커니즘은 attention 가중치와 Value값의 가중합을 통해 각 토큰별 가중치와 값을 구함

In [3]:
# Multi-Head Attention 계산
X = torch.tensor([[[1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0],
                   [0.0, 2.0, 0.0, 2.0, 0.0, 2.0, 0.0, 2.0],
                   [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]]])
print(f"X : {X.shape}")  # (B, T, F)

B, T, _ = X.shape
embedding_dim = 8
num_head = 4  # 헤드 갯수
heading_dim = embedding_dim // num_head  # 헤드당 차원(d_k = d_model/h)

W_q = nn.Linear(embedding_dim, embedding_dim, bias=False)
W_k = nn.Linear(embedding_dim, embedding_dim, bias=False)
W_v = nn.Linear(embedding_dim, embedding_dim, bias=False)

Q = W_q(X)  # X -> Q (배치, 길이, 차원)
K = W_k(X)
V = W_v(X)

print(f"Q : {Q.shape}")
print(f"K : {K.shape}")
print(f"V : {V.shape}")

# 헤드 분할
# B, T, embedding_dim
# -> B, T, num_head, heading_dim (view)
# -> B, num_head, T, heading_dim (transpose)
Q_head = Q.view(B, T, num_head, heading_dim).transpose(1, 2)  # Q를 헤드별로 분리(num_head) 후 차원 위치 변환
K_head = K.view(B, T, num_head, heading_dim).transpose(1, 2)
V_head = V.view(B, T, num_head, heading_dim).transpose(1, 2)

print(f"Q_head : {Q_head.shape}")  # (B, num_head, T, heading_dim)
print(f"K_head : {K_head.shape}")
print(f"V_head : {V_head.shape}")

# Q, K 유사도계산
# 각 헤드별로 Q•K^T 토큰간 유사도 계산 
attn_scores = torch.matmul(Q_head, K_head.transpose(-2, -1))
attn_scores /= embedding_dim ** 0.5  # 차원(d_k)로 나눈 후 score 스케일 조정(softmax값 안정화)
print(f"attn_scores : {attn_scores.shape}")

# Attention 분포 (확률)
attn_weights = F.softmax(attn_scores, dim=-1)  # 각 토큰이 바라볼 비율을 확률로 변환(행 단위 합)
print(f"attn_weights : {attn_weights.shape}")

# V-attention 분포의 가중합
output = torch.matmul(attn_weights, V_head)  # attention 가중치와 헤드별 Value의 가중합 = 최종 출력
print(f"attn_value: {output.shape}")

# 헤드 결합
output = output.transpose(1, 2)  # (B, num_head, T, d_k) -> (B, T, num_head, d_k)
# contiguous(): view() 호출전 메모리 연속된 상태변환 확인
output = output.contiguous().view(B, T, embedding_dim)  # (B, T, num_head*d_k) -> (B, T, model_d)
print(f"헤드결합 후 출력 : {output.shape}")

X : torch.Size([1, 3, 8])
Q : torch.Size([1, 3, 8])
K : torch.Size([1, 3, 8])
V : torch.Size([1, 3, 8])
Q_head : torch.Size([1, 4, 3, 2])
K_head : torch.Size([1, 4, 3, 2])
V_head : torch.Size([1, 4, 3, 2])
attn_scores : torch.Size([1, 4, 3, 3])
attn_weights : torch.Size([1, 4, 3, 3])
attn_value: torch.Size([1, 4, 3, 2])
헤드결합 후 출력 : torch.Size([1, 3, 8])
